# 🔐 Attaque par fautes sur le DES
## Calcul Sécurisé — M2 Cryptographie et Algèbre Appliquée

**Auteur :** Diaw Papa Amadou  
**Université :** UVSQ — Versailles Saint-Quentin-en-Yvelines  
**Date :** 2025-2026

---

## Objectif

Retrouver la clé secrète DES à partir de :
- 1 message clair
- 1 chiffré correct
- 32 chiffrés faux obtenus par injection de fautes sur R15

## Résultat obtenu

**Clé trouvée : `FE 76 5D 01 D0 54 9E 20`**

## 1 — Tables du DES
Les tables suivantes définissent toutes les permutations
et substitutions utilisées dans l'algorithme DES.

In [3]:
# ===== TABLES DU DES =====

# Permutation initiale
IP = [
    58, 50, 42, 34, 26, 18, 10, 2,
    60, 52, 44, 36, 28, 20, 12, 4,
    62, 54, 46, 38, 30, 22, 14, 6,
    64, 56, 48, 40, 32, 24, 16, 8,
    57, 49, 41, 33, 25, 17,  9, 1,
    59, 51, 43, 35, 27, 19, 11, 3,
    61, 53, 45, 37, 29, 21, 13, 5,
    63, 55, 47, 39, 31, 23, 15, 7
]

# Permutation finale
IP_INV = [
    40, 8, 48, 16, 56, 24, 64, 32,
    39, 7, 47, 15, 55, 23, 63, 31,
    38, 6, 46, 14, 54, 22, 62, 30,
    37, 5, 45, 13, 53, 21, 61, 29,
    36, 4, 44, 12, 52, 20, 60, 28,
    35, 3, 43, 11, 51, 19, 59, 27,
    34, 2, 42, 10, 50, 18, 58, 26,
    33, 1, 41,  9, 49, 17, 57, 25
]

# Expansion E (32 → 48 bits)
E = [
    32,  1,  2,  3,  4,  5,
     4,  5,  6,  7,  8,  9,
     8,  9, 10, 11, 12, 13,
    12, 13, 14, 15, 16, 17,
    16, 17, 18, 19, 20, 21,
    20, 21, 22, 23, 24, 25,
    24, 25, 26, 27, 28, 29,
    28, 29, 30, 31, 32,  1
]

# Permutation P
P = [
    16,  7, 20, 21, 29, 12, 28, 17,
     1, 15, 23, 26,  5, 18, 31, 10,
     2,  8, 24, 14, 32, 27,  3,  9,
    19, 13, 30,  6, 22, 11,  4, 25
]

# PC-1 (64 → 56 bits)
PC1 = [
    57, 49, 41, 33, 25, 17,  9,
     1, 58, 50, 42, 34, 26, 18,
    10,  2, 59, 51, 43, 35, 27,
    19, 11,  3, 60, 52, 44, 36,
    63, 55, 47, 39, 31, 23, 15,
     7, 62, 54, 46, 38, 30, 22,
    14,  6, 61, 53, 45, 37, 29,
    21, 13,  5, 28, 20, 12,  4
]

# PC-2 (56 → 48 bits)
PC2 = [
    14, 17, 11, 24,  1,  5,
     3, 28, 15,  6, 21, 10,
    23, 19, 12,  4, 26,  8,
    16,  7, 27, 20, 13,  2,
    41, 52, 31, 37, 47, 55,
    30, 40, 51, 45, 33, 48,
    44, 49, 39, 56, 34, 53,
    46, 42, 50, 36, 29, 32
]

# Décalages pour génération des sous-clés
DECALAGES = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1]

# 8 Boîtes S
S_BOXES = [
    # S1
    [
        [14,  4, 13,  1,  2, 15, 11,  8,  3, 10,  6, 12,  5,  9,  0,  7],
        [ 0, 15,  7,  4, 14,  2, 13,  1, 10,  6, 12, 11,  9,  5,  3,  8],
        [ 4,  1, 14,  8, 13,  6,  2, 11, 15, 12,  9,  7,  3, 10,  5,  0],
        [15, 12,  8,  2,  4,  9,  1,  7,  5, 11,  3, 14, 10,  0,  6, 13]
    ],
    # S2
    [
        [15,  1,  8, 14,  6, 11,  3,  4,  9,  7,  2, 13, 12,  0,  5, 10],
        [ 3, 13,  4,  7, 15,  2,  8, 14, 12,  0,  1, 10,  6,  9, 11,  5],
        [ 0, 14,  7, 11, 10,  4, 13,  1,  5,  8, 12,  6,  9,  3,  2, 15],
        [13,  8, 10,  1,  3, 15,  4,  2, 11,  6,  7, 12,  0,  5, 14,  9]
    ],
    # S3
    [
        [10,  0,  9, 14,  6,  3, 15,  5,  1, 13, 12,  7, 11,  4,  2,  8],
        [13,  7,  0,  9,  3,  4,  6, 10,  2,  8,  5, 14, 12, 11, 15,  1],
        [13,  6,  4,  9,  8, 15,  3,  0, 11,  1,  2, 12,  5, 10, 14,  7],
        [ 1, 10, 13,  0,  6,  9,  8,  7,  4, 15, 14,  3, 11,  5,  2, 12]
    ],
    # S4
    [
        [ 7, 13, 14,  3,  0,  6,  9, 10,  1,  2,  8,  5, 11, 12,  4, 15],
        [13,  8, 11,  5,  6, 15,  0,  3,  4,  7,  2, 12,  1, 10, 14,  9],
        [10,  6,  9,  0, 12, 11,  7, 13, 15,  1,  3, 14,  5,  2,  8,  4],
        [ 3, 15,  0,  6, 10,  1, 13,  8,  9,  4,  5, 11, 12,  7,  2, 14]
    ],
    # S5
    [
        [ 2, 12,  4,  1,  7, 10, 11,  6,  8,  5,  3, 15, 13,  0, 14,  9],
        [14, 11,  2, 12,  4,  7, 13,  1,  5,  0, 15, 10,  3,  9,  8,  6],
        [ 4,  2,  1, 11, 10, 13,  7,  8, 15,  9, 12,  5,  6,  3,  0, 14],
        [11,  8, 12,  7,  1, 14,  2, 13,  6, 15,  0,  9, 10,  4,  5,  3]
    ],
    # S6
    [
        [12,  1, 10, 15,  9,  2,  6,  8,  0, 13,  3,  4, 14,  7,  5, 11],
        [10, 15,  4,  2,  7, 12,  9,  5,  6,  1, 13, 14,  0, 11,  3,  8],
        [ 9, 14, 15,  5,  2,  8, 12,  3,  7,  0,  4, 10,  1, 13, 11,  6],
        [ 4,  3,  2, 12,  9,  5, 15, 10, 11, 14,  1,  7,  6,  0,  8, 13]
    ],
    # S7
    [
        [ 4, 11,  2, 14, 15,  0,  8, 13,  3, 12,  9,  7,  5, 10,  6,  1],
        [13,  0, 11,  7,  4,  9,  1, 10, 14,  3,  5, 12,  2, 15,  8,  6],
        [ 1,  4, 11, 13, 12,  3,  7, 14, 10, 15,  6,  8,  0,  5,  9,  2],
        [ 6, 11, 13,  8,  1,  4, 10,  7,  9,  5,  0, 15, 14,  2,  3, 12]
    ],
    # S8
    [
        [13,  2,  8,  4,  6, 15, 11,  1, 10,  9,  3, 14,  5,  0, 12,  7],
        [ 1, 15, 13,  8, 10,  3,  7,  4, 12,  5,  6, 11,  0, 14,  9,  2],
        [ 7, 11,  4,  1,  9, 12, 14,  2,  0,  6, 10, 13, 15,  3,  5,  8],
        [ 2,  1, 14,  7,  4, 10,  8, 13, 15, 12,  9,  0,  3,  5,  6, 11]
    ]
]

print("Tables DES chargées !")
print(f"  IP     : {len(IP)} éléments")
print(f"  IP_INV : {len(IP_INV)} éléments")
print(f"  E      : {len(E)} éléments")
print(f"  P      : {len(P)} éléments")
print(f"  PC1    : {len(PC1)} éléments")
print(f"  PC2    : {len(PC2)} éléments")
print(f"  S_BOXES: {len(S_BOXES)} boîtes")

Tables DES chargées !
  IP     : 64 éléments
  IP_INV : 64 éléments
  E      : 48 éléments
  P      : 32 éléments
  PC1    : 56 éléments
  PC2    : 48 éléments
  S_BOXES: 8 boîtes


## 2 — Fonctions utilitaires
Fonctions de base pour manipuler les bits.

In [4]:
# ===== FONCTIONS UTILITAIRES =====

def int_vers_bits(n, longueur):
    """
    Convertit un entier en liste de bits
    Exemple : int_vers_bits(6, 8) → [0,0,0,0,0,1,1,0]
    """
    bits = []
    for i in range(longueur-1, -1, -1):
        bits.append((n >> i) & 1)
    return bits


def bits_vers_int(bits):
    """
    Convertit une liste de bits en entier
    Exemple : bits_vers_int([0,0,0,0,0,1,1,0]) → 6
    """
    n = 0
    for bit in bits:
        n = (n << 1) | bit
    return n


def permuter(bits, table):
    """
    Applique une permutation sur une liste de bits
    Exemple : permuter([a,b,c,d], [3,1,4,2]) → [c,a,d,b]
    """
    return [bits[i-1] for i in table]


def hex_vers_bits(liste_hex):
    """
    Convertit une liste d'octets hexa en liste de bits
    Exemple : [0x86, 0x67] → [1,0,0,0,0,1,1,0, 0,1,1,0,0,1,1,1]
    """
    bits = []
    for octet in liste_hex:
        bits += int_vers_bits(octet, 8)
    return bits


# ===== TESTS =====
print(" Fonctions utilitaires chargées !")
print(f"  int_vers_bits(0x86, 8) = {int_vers_bits(0x86, 8)}")
print(f"  bits_vers_int([1,0,0,0,0,1,1,0]) = {hex(bits_vers_int([1,0,0,0,0,1,1,0]))}")

 Fonctions utilitaires chargées !
  int_vers_bits(0x86, 8) = [1, 0, 0, 0, 0, 1, 1, 0]
  bits_vers_int([1,0,0,0,0,1,1,0]) = 0x86


## 3 — Génération des sous-clés

In [6]:
# ===== GÉNÉRATION DES SOUS-CLÉS =====

def rotation_gauche(bits, n):
    """Rotation vers la gauche de n positions"""
    return bits[n:] + bits[:n]


def generer_sous_cles(cle_64bits):
    """
    Génère les 16 sous-clés de 48 bits depuis la clé principale
    
    Étapes :
    1. PC-1 : 64 → 56 bits (enlève les bits de parité)
    2. Diviser en C et D de 28 bits
    3. Pour chaque tour : rotation + PC-2 → Ki (48 bits)
    """
    # PC-1 : 64 → 56 bits
    cle_56 = permuter(cle_64bits, PC1)
    
    # Diviser en deux moitiés
    C = cle_56[:28]
    D = cle_56[28:]
    
    # Générer les 16 sous-clés
    sous_cles = []
    for i in range(16):
        C = rotation_gauche(C, DECALAGES[i])
        D = rotation_gauche(D, DECALAGES[i])
        Ki = permuter(C + D, PC2)
        sous_cles.append(Ki)
    
    return sous_cles


print(" Génération des sous-clés chargée !")

 Génération des sous-clés chargée !


## 4 — Fonction F et DES complet

In [7]:
# ===== FONCTION F =====

def expansion_E(R):
    """Expansion : 32 → 48 bits"""
    return permuter(R, E)


def calculer_sortie_S(entree, i):
    """
    Calcule la sortie de la boîte S numéro i
    entree : 6 bits
    retourne : 4 bits
    """
    ligne   = (entree[0] << 1) | entree[5]
    colonne = (entree[1] << 3) | (entree[2] << 2) | (entree[3] << 1) | entree[4]
    valeur  = S_BOXES[i][ligne][colonne]
    return int_vers_bits(valeur, 4)


def boites_S(bits_48):
    """8 boîtes S : 48 → 32 bits"""
    resultat = []
    for i in range(8):
        groupe  = bits_48[i*6 : i*6+6]
        resultat += calculer_sortie_S(groupe, i)
    return resultat


def fonction_F(R, K):
    """
    Fonction de tour F
    R : 32 bits, K : 48 bits → 32 bits
    """
    R_etendu   = expansion_E(R)
    xor_result = [R_etendu[i] ^ K[i] for i in range(48)]
    apres_S    = boites_S(xor_result)
    return permuter(apres_S, P)


# ===== DES COMPLET =====

def DES_chiffrer(message_hex, cle_hex):
    """
    Chiffrement DES complet
    message_hex : liste de 8 octets
    cle_hex     : liste de 8 octets
    retourne    : liste de 8 octets chiffrés
    """
    # Convertir en bits
    message_bits = hex_vers_bits(message_hex)
    cle_bits     = hex_vers_bits(cle_hex)
    
    # Permutation initiale IP
    apres_IP = permuter(message_bits, IP)
    L = apres_IP[:32]
    R = apres_IP[32:]
    
    # Générer les 16 sous-clés
    sous_cles = generer_sous_cles(cle_bits)
    
    # 16 tours Feistel
    for i in range(16):
        L_nouveau = R
        R_nouveau = [L[j] ^ fonction_F(R, sous_cles[i])[j] for j in range(32)]
        L = L_nouveau
        R = R_nouveau
    
    # Permutation finale IP_INV (R avant L !)
    chiffre_bits = permuter(R + L, IP_INV)
    
    # Convertir en octets
    chiffre_hex = []
    for i in range(8):
        chiffre_hex.append(bits_vers_int(chiffre_bits[i*8 : i*8+8]))
    
    return chiffre_hex


print(" Fonction F et DES complet chargés !")

 Fonction F et DES complet chargés !


## 5 — Données du problème

In [8]:
# ===== DONNÉES =====

# Message clair
message_clair = [0x86, 0x67, 0xA7, 0x0B, 0x08, 0xE6, 0x5B, 0x61]

# Chiffré juste (sans faute)
chiffre_juste = [0x61, 0xD5, 0x83, 0x36, 0xBF, 0xD4, 0x83, 0xB0]

# 32 chiffrés faux (avec fautes sur R15)
chiffres_faux = [
    [0x61, 0x95, 0x97, 0x3E, 0xEF, 0xC4, 0x83, 0xB3],
    [0x21, 0xD5, 0xA3, 0x36, 0xBB, 0x94, 0x8A, 0xF1],
    [0x65, 0x55, 0xC2, 0x26, 0xBE, 0xF4, 0x83, 0xB0],
    [0x63, 0xD0, 0x83, 0x32, 0x3E, 0xD5, 0xC3, 0xA4],
    [0x61, 0xC5, 0x83, 0x71, 0xBF, 0xC4, 0x83, 0xB2],
    [0x21, 0xD5, 0x8B, 0x37, 0xBF, 0x90, 0x8B, 0xB1],
    [0x65, 0xF5, 0xC2, 0x36, 0xBF, 0xF4, 0x93, 0xF0],
    [0xF5, 0xD4, 0x83, 0x26, 0x3E, 0xD4, 0x87, 0xB4],
    [0x61, 0x84, 0x83, 0x74, 0xAF, 0xD5, 0x83, 0x24],
    [0x60, 0x95, 0x8F, 0x37, 0xAF, 0xD0, 0x81, 0xB0],
    [0x61, 0xF5, 0x93, 0x36, 0xBF, 0xDC, 0x93, 0xF1],
    [0xF1, 0xD5, 0x82, 0x26, 0x9B, 0xD4, 0xC6, 0xF0],
    [0x61, 0xD1, 0x83, 0xA2, 0xBE, 0xD5, 0x87, 0x20],
    [0x60, 0x85, 0x85, 0x76, 0xAF, 0xD4, 0x81, 0xB0],
    [0x61, 0xDD, 0x93, 0x37, 0xBF, 0xCC, 0x83, 0xB1],
    [0x05, 0xD5, 0xD3, 0x36, 0x9B, 0x94, 0x93, 0xB1],
    [0x75, 0xD4, 0xC3, 0xB6, 0xBF, 0xD4, 0x83, 0x84],
    [0x61, 0xC0, 0x81, 0x76, 0xBF, 0xD4, 0x03, 0xB4],
    [0x61, 0x9D, 0x87, 0x37, 0xBF, 0xC6, 0x83, 0xB0],
    [0x01, 0xD5, 0x83, 0x37, 0xB7, 0xD0, 0x92, 0xF1],
    [0x71, 0xD5, 0x82, 0x06, 0xBB, 0xD4, 0xC7, 0xD0],
    [0x61, 0xD0, 0x03, 0x26, 0xBF, 0xD4, 0x07, 0xB4],
    [0x61, 0x97, 0x87, 0x76, 0xBF, 0xD7, 0x83, 0xB0],
    [0x28, 0x95, 0x87, 0x36, 0xA7, 0x84, 0x83, 0xB0],
    [0x75, 0xD5, 0x93, 0x16, 0xBB, 0x94, 0x82, 0xB9],
    [0x75, 0xD5, 0x03, 0x26, 0xBF, 0xD4, 0xA7, 0xB0],
    [0x61, 0xD6, 0x83, 0x76, 0xBF, 0x55, 0x83, 0xA0],
    [0x69, 0x85, 0x87, 0x33, 0xFD, 0xD0, 0x83, 0xB0],
    [0x21, 0xD5, 0x93, 0x3F, 0xFB, 0x94, 0x93, 0xB8],
    [0x75, 0xD5, 0xA3, 0x36, 0xBB, 0xD4, 0xA2, 0xF0],
    [0x61, 0x54, 0x83, 0x26, 0xBE, 0x54, 0x83, 0xA0],
    [0x63, 0x84, 0x83, 0x76, 0xAD, 0xD4, 0x83, 0xB4]
]

print("Données chargées !")
print(f"  Message clair  : {[hex(x) for x in message_clair]}")
print(f"  Chiffré juste  : {[hex(x) for x in chiffre_juste]}")
print(f"  Chiffrés faux  : {len(chiffres_faux)} chiffrés")

Données chargées !
  Message clair  : ['0x86', '0x67', '0xa7', '0xb', '0x8', '0xe6', '0x5b', '0x61']
  Chiffré juste  : ['0x61', '0xd5', '0x83', '0x36', '0xbf', '0xd4', '0x83', '0xb0']
  Chiffrés faux  : 32 chiffrés


## 6 — Attaque par fautes
### Extraction de R15 et R16

In [9]:
# ===== EXTRACTION R15 ET R16 =====

def extraire_R15_R16(chiffre_hex):
    """
    Depuis un chiffré, extrait R15 et R16
    
    Structure du dernier tour :
        L16 = R15
        R16 = L15 ⊕ F(R15, K16)
        C   = IP_INV(R16 + L16)
    
    Donc :
        IP(C) = R16 + L16
        R16   = IP(C)[:32]
        R15   = L16 = IP(C)[32:]
    """
    bits     = hex_vers_bits(chiffre_hex)
    apres_IP = permuter(bits, IP)
    R16      = apres_IP[:32]
    R15      = apres_IP[32:]
    return R15, R16


# Test
R15, R16 = extraire_R15_R16(chiffre_juste)
print(" Extraction R15/R16 !")
print(f"  R15 ({len(R15)} bits) : {R15}")
print(f"  R16 ({len(R16)} bits) : {R16}")

 Extraction R15/R16 !
  R15 (32 bits) : [1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0]
  R16 (32 bits) : [0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1]


### Attaque sur les boîtes S

In [11]:
# ===== ATTAQUE SUR LES BOÎTES S =====

def attaque_une_boite_S(numero_S, R15, R15_etoile, R16, R16_etoile):
    """
    Attaque sur la boîte S numéro numero_S (0 à 7)
    
    Principe :
        R16 ⊕ R16* = F(R15, K16) ⊕ F(R15*, K16)
    
    Pour chaque candidat k (0 à 63) :
        - Calculer sortie_S(R15,  k) ⊕ sortie_S(R15*, k)
        - Comparer avec différence observée dans R16
        - Si différent → éliminer k
    
    Retourne : ensemble des candidats qui survivent
    """
    # Expansion de R15 et R15*
    R15_exp  = expansion_E(R15)
    R15e_exp = expansion_E(R15_etoile)
    
    # Extraire les 6 bits pour la boîte S i
    debut      = numero_S * 6
    R15_6bits  = R15_exp [debut : debut+6]
    R15e_6bits = R15e_exp[debut : debut+6]
    
    # Inverser P pour isoler la contribution de Si
    P_INV = [0] * 32
    for j in range(32):
        P_INV[P[j]-1] = j
    
    positions_Si = [P_INV[numero_S*4 + j] for j in range(4)]
    
    # Différence observée dans R16
    diff_R16 = [R16[j] ^ R16_etoile[j] for j in range(32)]
    diff_Si  = [diff_R16[positions_Si[j]] for j in range(4)]
    
    # Tester les 64 candidats
    candidats = set(range(64))
    
    for k in range(64):
        k_bits   = int_vers_bits(k, 6)
        entree_1 = [R15_6bits[j]  ^ k_bits[j] for j in range(6)]
        entree_2 = [R15e_6bits[j] ^ k_bits[j] for j in range(6)]
        
        sortie_1  = calculer_sortie_S(entree_1, numero_S)
        sortie_2  = calculer_sortie_S(entree_2, numero_S)
        diff_calc = [sortie_1[j] ^ sortie_2[j] for j in range(4)]
        
        if diff_calc != diff_Si:
            candidats.discard(k)
    
    return candidats


print(" Fonction attaque_une_boite_S chargée !")

 Fonction attaque_une_boite_S chargée !


### Attaque complète — Résultats K16

In [12]:
# ===== ATTAQUE COMPLÈTE =====

def attaque_complete(chiffre_juste, chiffres_faux):
    """
    Attaque par fautes complète sur les 8 boîtes S
    Retourne K16 (48 bits)
    """
    print("=== Attaque par fautes sur le DES ===\n")
    
    # Extraire R15 et R16 depuis le chiffré juste
    R15, R16 = extraire_R15_R16(chiffre_juste)
    
    # Initialiser candidats
    candidats = [set(range(64)) for _ in range(8)]
    
    # Pour chaque chiffré faux
    for idx, chiffre_faux in enumerate(chiffres_faux):
        R15_f, R16_f = extraire_R15_R16(chiffre_faux)
        
        # Ignorer les fautes nulles
        if R15_f == R15:
            continue
        
        # Attaquer chaque boîte S
        for i in range(8):
            nouveaux      = attaque_une_boite_S(i, R15, R15_f, R16, R16_f)
            candidats[i]  = candidats[i] & nouveaux
        
        nb = [len(candidats[i]) for i in range(8)]
        print(f"Chiffré {idx+1:2d} : candidats = {nb}")
    
    # Résultats
    print("\n=== Résultats par boîte S ===")
    K16_bits    = []
    valeurs_K16 = []
    
    for i in range(8):
        valeur = list(candidats[i])
        print(f"S{i+1} : {len(candidats[i])} candidat(s) → {valeur}")
        v = valeur[0]
        valeurs_K16.append(v)
        K16_bits += int_vers_bits(v, 6)
    
    print(f"\n K16 trouvé !")
    print(f"Valeurs : {valeurs_K16}")
    print(f"K16 (48 bits) : {K16_bits}")
    
    return K16_bits, valeurs_K16


# Lancer l'attaque
K16_bits, valeurs_K16 = attaque_complete(chiffre_juste, chiffres_faux)

=== Attaque par fautes sur le DES ===

Chiffré  1 : candidats = [64, 64, 64, 64, 8, 4, 6, 64]
Chiffré  2 : candidats = [64, 64, 64, 8, 1, 4, 6, 64]
Chiffré  3 : candidats = [64, 10, 4, 8, 1, 4, 6, 64]
Chiffré  4 : candidats = [10, 1, 4, 8, 1, 4, 6, 10]
Chiffré  5 : candidats = [10, 1, 4, 8, 1, 4, 1, 1]
Chiffré  6 : candidats = [10, 1, 4, 8, 1, 1, 1, 1]
Chiffré  7 : candidats = [10, 1, 4, 2, 1, 1, 1, 1]
Chiffré  8 : candidats = [1, 1, 2, 2, 1, 1, 1, 1]
Chiffré  9 : candidats = [1, 1, 2, 2, 1, 1, 1, 1]
Chiffré 10 : candidats = [1, 1, 2, 2, 1, 1, 1, 1]
Chiffré 11 : candidats = [1, 1, 2, 2, 1, 1, 1, 1]
Chiffré 12 : candidats = [1, 1, 2, 1, 1, 1, 1, 1]
Chiffré 13 : candidats = [1, 1, 2, 1, 1, 1, 1, 1]
Chiffré 14 : candidats = [1, 1, 2, 1, 1, 1, 1, 1]
Chiffré 15 : candidats = [1, 1, 2, 1, 1, 1, 1, 1]
Chiffré 16 : candidats = [1, 1, 1, 1, 1, 1, 1, 1]
Chiffré 17 : candidats = [1, 1, 1, 1, 1, 1, 1, 1]
Chiffré 18 : candidats = [1, 1, 1, 1, 1, 1, 1, 1]
Chiffré 19 : candidats = [1, 1, 1, 1, 1, 1, 

## 7 — Retrouver la clé complète
### Étape 1 : Inverser PC-2

In [13]:
# ===== INVERSER PC-2 =====

def inverser_PC2(K16_bits):
    """
    Retrouve les 56 bits partiels depuis K16 (48 bits)
    en inversant PC-2
    
    PC-2 sélectionne 48 bits parmi 56
    → 8 bits restent inconnus
    """
    # Initialiser 56 positions à None
    cle_56 = [None] * 56
    
    # Replacer chaque bit de K16 à sa position d'origine
    for i, pos in enumerate(PC2):
        cle_56[pos-1] = K16_bits[i]
    
    # Trouver les positions manquantes
    positions_manquantes = [i+1 for i in range(56) if cle_56[i] is None]
    
    print(f"Bits connus      : {56 - len(positions_manquantes)}")
    print(f"Bits inconnus    : {len(positions_manquantes)}")
    print(f"Positions manquantes : {positions_manquantes}")
    
    return cle_56, positions_manquantes


# Inverser PC-2
cle_56_partielle, positions_manquantes = inverser_PC2(K16_bits)
print(f"\nClé 56 bits partielle : {cle_56_partielle}")

Bits connus      : 48
Bits inconnus    : 8
Positions manquantes : [9, 18, 22, 25, 35, 38, 43, 54]

Clé 56 bits partielle : [0, 1, 0, 1, 0, 0, 0, 1, None, 0, 1, 1, 0, 1, 1, 1, 1, None, 0, 0, 0, None, 1, 1, None, 1, 1, 1, 0, 1, 0, 0, 0, 0, None, 1, 0, None, 1, 0, 0, 1, None, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, None, 1, 1]


### Étape 2 : Inverser PC-1

In [14]:
# ===== INVERSER PC-1 =====

def inverser_PC1(cle_56):
    """
    Reconstruit la clé 64 bits depuis 56 bits
    en inversant PC-1
    """
    cle_64_bits = [0] * 64
    
    # Replacer chaque bit à sa position d'origine
    for i, pos in enumerate(PC1):
        cle_64_bits[pos-1] = cle_56[i]
    
    # Convertir en octets
    cle_64_octets = []
    for i in range(8):
        octet_bits = cle_64_bits[i*8 : i*8+8]
        cle_64_octets.append(bits_vers_int(octet_bits))
    
    return cle_64_octets

print("Fonction inverser_PC1 chargée !")

Fonction inverser_PC1 chargée !


### Étape 3 : Force brute sur 8 bits manquants

In [15]:
# ===== FORCE BRUTE =====

def force_brute(cle_56_partielle, positions_manquantes, message, chiffre_juste):
    """
    Force brute sur les 8 bits manquants
    Teste 2^8 = 256 combinaisons
    """
    print("=== Force brute sur 8 bits manquants ===")
    print(f"Nombre de combinaisons : 2^8 = 256\n")
    
    for combinaison in range(256):
        # Convertir en 8 bits
        bits_manquants = int_vers_bits(combinaison, 8)
        
        # Remplir les positions manquantes
        cle_56 = cle_56_partielle.copy()
        for i, pos in enumerate(positions_manquantes):
            cle_56[pos-1] = bits_manquants[i]
        
        # Inverser PC-1 → clé 64 bits
        cle_64 = inverser_PC1(cle_56)
        
        # Tester le DES
        chiffre_test = DES_chiffrer(message, cle_64)
        
        # Vérifier
        if chiffre_test == chiffre_juste:
            print(f"Clé trouvée ! (combinaison {combinaison})")
            print(f"Clé 64 bits (hex) : {[hex(x) for x in cle_64]}")
            return cle_56, cle_64
    
    print("❌ Aucune clé trouvée !")
    return None, None


# Lancer la force brute
cle_56, cle_64 = force_brute(
    cle_56_partielle,
    positions_manquantes,
    message_clair,
    chiffre_juste
)

=== Force brute sur 8 bits manquants ===
Nombre de combinaisons : 2^8 = 256

Clé trouvée ! (combinaison 15)
Clé 64 bits (hex) : ['0xfe', '0x76', '0x5c', '0x0', '0xd0', '0x54', '0x9e', '0x20']


### Étape 4 : Ajouter les bits de parité

In [16]:
# ===== BITS DE PARITÉ =====

def ajouter_parite(cle_64_octets):
    """
    Ajoute les bits de parité à la clé 64 bits
    Chaque octet doit avoir un nombre impair de 1
    Le bit de poids faible est le bit de parité
    """
    print("=== Ajout des bits de parité ===\n")
    
    cle_finale = []
    
    for i, octet in enumerate(cle_64_octets):
        # Extraire les 7 bits de données
        bits_7 = int_vers_bits(octet, 8)[:7]
        
        # Compter les 1
        nb_uns = sum(bits_7)
        
        # Bit de parité pour avoir un nombre impair de 1
        bit_parite = 0 if nb_uns % 2 == 1 else 1
        
        # Construire l'octet final
        octet_final = bits_vers_int(bits_7 + [bit_parite])
        cle_finale.append(octet_final)
        
        print(f"Octet {i+1} : {bits_7} + [{bit_parite}] "
              f"= {hex(octet_final).upper()} "
              f"({nb_uns + bit_parite} uns → impair )")
    
    return cle_finale


# Ajouter la parité
cle_finale = ajouter_parite(cle_64)
print(f"\nClé finale : "
      f"{' '.join([hex(x)[2:].upper().zfill(2) for x in cle_finale])}")

=== Ajout des bits de parité ===

Octet 1 : [1, 1, 1, 1, 1, 1, 1] + [0] = 0XFE (7 uns → impair )
Octet 2 : [0, 1, 1, 1, 0, 1, 1] + [0] = 0X76 (5 uns → impair )
Octet 3 : [0, 1, 0, 1, 1, 1, 0] + [1] = 0X5D (5 uns → impair )
Octet 4 : [0, 0, 0, 0, 0, 0, 0] + [1] = 0X1 (1 uns → impair )
Octet 5 : [1, 1, 0, 1, 0, 0, 0] + [0] = 0XD0 (3 uns → impair )
Octet 6 : [0, 1, 0, 1, 0, 1, 0] + [0] = 0X54 (3 uns → impair )
Octet 7 : [1, 0, 0, 1, 1, 1, 1] + [0] = 0X9E (5 uns → impair )
Octet 8 : [0, 0, 1, 0, 0, 0, 0] + [0] = 0X20 (1 uns → impair )

Clé finale : FE 76 5D 01 D0 54 9E 20


## 8 — Vérification finale

In [17]:
# ===== VÉRIFICATION FINALE =====

print("=== Vérification finale ===\n")

# Chiffrer avec la clé trouvée
chiffre_verification = DES_chiffrer(message_clair, cle_finale)

print(f"Message clair   : "
      f"{' '.join([hex(x)[2:].upper().zfill(2) for x in message_clair])}")
print(f"Clé trouvée     : "
      f"{' '.join([hex(x)[2:].upper().zfill(2) for x in cle_finale])}")
print(f"Chiffré obtenu  : "
      f"{' '.join([hex(x)[2:].upper().zfill(2) for x in chiffre_verification])}")
print(f"Chiffré attendu : "
      f"{' '.join([hex(x)[2:].upper().zfill(2) for x in chiffre_juste])}")

if chiffre_verification == chiffre_juste:
    print("\n🎉 SUCCÈS ! La clé est correcte !")
    print(f"\nClé finale : "
          f"{' '.join([hex(x)[2:].upper().zfill(2) for x in cle_finale])}")
else:
    print("\n❌ ERREUR ! La clé est incorrecte !")

=== Vérification finale ===

Message clair   : 86 67 A7 0B 08 E6 5B 61
Clé trouvée     : FE 76 5D 01 D0 54 9E 20
Chiffré obtenu  : 61 D5 83 36 BF D4 83 B0
Chiffré attendu : 61 D5 83 36 BF D4 83 B0

🎉 SUCCÈS ! La clé est correcte !

Clé finale : FE 76 5D 01 D0 54 9E 20
